In [0]:
%python

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType,
    DoubleType, DecimalType, ArrayType
)

xml_file = '/mnt/dplandingstoragetest/brd_example/data/aggregated_generation_2050.xml'

interval_schema = StructType([
    StructField("start", TimestampType(), False),
    StructField("end", TimestampType(), False)
])  

point_schema = StructType([
    StructField("position", IntegerType(), False),
    StructField("quantity", DecimalType(15, 6), True) 
])

period_schema = StructType([
    StructField("timeInterval", StructType(interval_schema), False),  # -> interval_schema
    StructField("resolution", StringType(), False),
    StructField("Point", ArrayType(point_schema), False)
])

timeseries_schema = StructType([
    StructField("mRID", StringType(), False),
    StructField("businessType", StringType(), False),
    StructField("objectAggregation", StringType(), False),
    StructField("inBiddingZone_Domain.mRID", StringType(), True),
    StructField("quantity_Measure_Unit.name", StringType(), True),
    StructField("curveType", StringType(), False),
    StructField("MktPSRType", StructType([StructField("psrType", StringType(), True)]), True),
    StructField("Period", ArrayType(period_schema), False)
])

gl_market_schema = StructType([
    StructField("mRID", StringType(), False),
    StructField("revisionNumber", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("process.processType", StringType(), False),
    StructField("sender_MarketParticipant.mRID", StringType(), False),
    StructField("sender_MarketParticipant.marketRole.type", StringType(), False),
    StructField("receiver_MarketParticipant.mRID", StringType(), False),
    StructField("receiver_MarketParticipant.marketRole.type", StringType(), False),
    StructField("createdDateTime", TimestampType(), False),
    StructField("time_Period.timeInterval", StructType(interval_schema), False),   # -> interval_schema
    StructField("TimeSeries", ArrayType(timeseries_schema), False)
])

df = spark.read.format("com.databricks.spark.xml") \
    .option("rowTag", "GL_MarketDocument") \
    .schema(gl_market_schema) \
    .load(xml_file)

df.createOrReplaceTempView("gl_market")

In [0]:
select * from gl_market

In [0]:
CREATE OR REPLACE TEMPORARY VIEW new_data
AS
SELECT m.mRID 
     , m.revisionNumber             
     , m.type
     , m.`process.processType`                        as process_processType
     , m.`sender_MarketParticipant.mRID`              as sender_MarketParticipant_mRID
     , m.`sender_MarketParticipant.marketRole.type`   as sender_MarketParticipant_marketRole_type
     , m.`receiver_MarketParticipant.mRID`            as receiver_MarketParticipant_mRID
     , m.`receiver_MarketParticipant.marketRole.type` as receiver_MarketParticipant_marketRole_type
     , m.createdDateTime            
     , m.`time_Period.timeInterval`.start             as time_Period_timeInterval_start
     , m.`time_Period.timeInterval`.end               as time_Period_timeInterval_end
     , ts.mRID                                        as TimeSeries_mRID
     , ts.businessType                                as TimeSeries_businessType
     , ts.objectAggregation                           as TimeSeries_objectAggregation
     , ts.`inBiddingZone_Domain.mRID`                 as TimeSeries_inBiddingZone_Domain_mRID
     , ts.`quantity_Measure_Unit.name`                as TimeSeries_quantity_Measure_Unit_name
     , ts.curveType                                   as TimeSeries_curveType
     , ts.MktPSRType.psrType                          as TimeSeries_MktPSRType_psrType
     , p.timeInterval.start                           as TimeSeries_Period_timeInterval_start    
     , p.timeInterval.end                             as TimeSeries_Period_timeInterval_end
     , p.resolution                                   as TimeSeries_Period_resolution
     , pt.position                                    as TimeSeries_Period_Point_position 
     , pt.quantity                                    as TimeSeries_Period_Point_quantity
 
FROM gl_market m
LATERAL VIEW explode(m.TimeSeries) ts_table AS ts
LATERAL VIEW explode(ts.Period) p_table AS p
LATERAL VIEW explode(p.Point) pt_table AS pt;